# הרצת תיקון הגבול מהענף המעודכן בגיט

תחילה מעלים לגיט את `src/sim/env.py` המתוקן ואת `tools/boundary_fix` מהחבילה החדשה.
המחברת מורידה את הענף `boundary-capture-fix`; היא **אינה** מחילה שוב את התיקון.
לא משתמשים בה במקביל למחברת הישנה שמורידה את ה-commit הישן ומתקנת אותו מחדש.

הרץ את תאי ההתחלה לפי הסדר. ההרצה הראשונה משווה 20 תרחישים חדשים.
השחזור של התרחישים הישנים כבוי כברירת מחדל ודורש שני נתיבים שמורים.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, uuid

drive.mount('/content/drive')
BRANCH = 'boundary-capture-fix'  # Change only if you saved/merged the fix to another branch.
REPO_URL = 'https://github.com/ohadnir31-cmyk/dynamic-interception-sim.git'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
REPO = Path('/content') / ('dynamic_interception_fixed_' + RUN_ID)
RESULTS = Path('/content/drive/MyDrive/dynamic_interception_outputs') / ('git_boundary_' + RUN_ID)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements.txt'), 'pytest'], check=True)
print('CHECKOUT:', REPO)
print('BRANCH:', BRANCH)
print('COMMIT:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
print('RESULTS:', RESULTS)
assert (REPO/'tools/boundary_fix/run_paired_check.py').is_file(), 'Upload the tools folder to this branch first.'


## בדיקות והשוואה קטנה
הפקודה כוללת בדיקות קוד והרצת חמש היוריסטיקות בשתי הגרסאות.
אין יצירת תוויות ואין אימון. פלט ישן אינו נדרס.


In [ ]:
PAIR_OUTPUT = RESULTS / ('pilot_20_' + uuid.uuid4().hex[:6])
subprocess.run([
    sys.executable, 'tools/boundary_fix/run_paired_check.py',
    '--repo', str(REPO), '--n-scenarios', '20', '--seed', '20260906',
    '--output-dir', str(PAIR_OUTPUT),
], cwd=REPO, check=True)


In [ ]:
import pandas as pd
from IPython.display import display
print(PAIR_OUTPUT/'boundary_fix_comparison.csv')
display(pd.read_csv(PAIR_OUTPUT/'boundary_fix_comparison.csv'))


## שחזור התרחישים הישנים - לא חובה להרצה הראשונה
מגדירים את התיקייה שבה נשמרו קובצי הניסוי המקורי ומפעילים את הדגל.
ברירת המחדל כאן היא 100 תרחישים. אחרי אימות תקינות, `LIMIT = 0` מריץ את כולם.
הכלי דורש התאמה לציונים הישנים לפני המשך לגרסה המתוקנת. מספרים חדשים אינם מחליפים תוויות או אימון.


In [ ]:
RUN_OLD_SCENARIOS = False
OLD_RESULTS = Path('/content/drive/MyDrive/REPLACE_WITH_YOUR_OLD_RESULTS_FOLDER')
LIMIT = 100  # 0 = all saved scenarios

if RUN_OLD_SCENARIOS:
    params = OLD_RESULTS/'large_scale_scenario_params.csv'
    scores = OLD_RESULTS/'large_scale_full_heuristic_rollouts.csv'
    if not params.is_file() or not scores.is_file():
        raise FileNotFoundError('Set OLD_RESULTS to the folder holding BOTH original CSV files.')
    OLD_REPLAY_OUTPUT = RESULTS / ('saved_' + str(LIMIT) + '_' + uuid.uuid4().hex[:6])
    subprocess.run([
        sys.executable, 'tools/boundary_fix/run_paired_check.py', '--repo', str(REPO),
        '--params-csv', str(params), '--original-results', str(scores), '--limit', str(LIMIT),
        '--output-dir', str(OLD_REPLAY_OUTPUT),
    ], cwd=REPO, check=True)
    display(pd.read_csv(OLD_REPLAY_OUTPUT/'boundary_fix_comparison.csv'))
    print('RESULTS:', OLD_REPLAY_OUTPUT)
else:
    print('Old-scenario replay is disabled.')


## מגבלות
קובץ התיקון נבדק בבדיקות ממוקדות; כלי ההשוואה נבדק בנתוני בדיקה.
המאגר המלא והנתונים שלך לא הורצו בסביבת ההכנה. בדיקות המאגר מתבצעות כאן בזמן ההפעלה.
תא שנכשל מחייב בירור: אין להסיר את בדיקות ההתאמה. מקור הקוד והסביבה נשמרים בתיקיית התוצאות.
המחברת אינה דוחפת שינויים לגיט ואינה מאמנת מודל.
